# TCGA Gene Expression + Reactome Graph Preparation (for BINN-as-GNN)

## Main goal

Prepare **cached, reusable artifacts** for graph-based learning on TCGA gene expression using Reactome pathways.

This notebook does **data + graph preparation only**. It intentionally stops before model training.

---

## What we download

### TCGA (UCSC Xena Toil hub)
- **Expression**: `tcga_RSEM_gene_tpm.gz` (genes as rows, samples as columns)
- **Phenotype**: `TcgaTargetGTEX_phenotype.txt.gz` (sample metadata, includes `_sample_type`)

### Reactome
- **Gene-to-pathway mapping**: `Ensembl2Reactome_All_Levels.txt`
- **Pathway hierarchy**: `ReactomePathwaysRelation.txt`

---

## What we build and save

### Dataset artifacts
- `outputs/expr_reactome_tcga_tumor_normal.parquet`  
  Expression filtered to Reactome-mapped genes and labeled TCGA samples

- `outputs/y_tcga_tumor_normal.csv`  
  Labels aligned to the expression columns

### Graph artifacts (saved under `outputs/graph/`)
- `edge_index.pt` (shape `[2, E]`, directed edges child → parent)
- `is_gene.pt` (boolean mask, length `N`)
- `node_table.csv` (node_id, node_name, node_type)
- `edge_table.csv` (src_name, dst_name, edge_type)
- `node2id.json` (mapping node_name → node_id)
- `metadata.json` (counts, filenames, parameters)

These are the only files you need to load later to start training a BINN-like GNN or a standard GNN.

---

## First task used here

We start with a simple binary classification label:

- `Primary Tumor` → 1  
- `Solid Tissue Normal` → 0

This label is highly imbalanced in TCGA. That is expected. We handle class imbalance later during training.


## 0. Environment setup

If you are running in Colab, we mount Google Drive so large files persist across sessions.


In [6]:
# Detect Colab and mount Drive (if available)
from pathlib import Path
import os

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    # Mount only if not already mounted
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    else:
        print("Drive already mounted.")
else:
    print("Not running in Colab. Skipping Drive mount.")

# Choose where to store data and outputs
BASE_DIR = Path("/content/drive/MyDrive/binn_gnn_tcga") if IN_COLAB else Path("./binn_gnn_tcga")
DATA_DIR = BASE_DIR / "data"
OUT_DIR = BASE_DIR / "outputs"
GRAPH_DIR = OUT_DIR / "graph"

for d in [DATA_DIR, OUT_DIR, GRAPH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)
print("GRAPH_DIR:", GRAPH_DIR)


Drive already mounted.
BASE_DIR: /content/drive/MyDrive/binn_gnn_tcga
DATA_DIR: /content/drive/MyDrive/binn_gnn_tcga/data
OUT_DIR: /content/drive/MyDrive/binn_gnn_tcga/outputs
GRAPH_DIR: /content/drive/MyDrive/binn_gnn_tcga/outputs/graph


In [7]:
# Install lightweight dependencies (no model training yet)
!pip -q install pandas numpy tqdm networkx pyarrow

import re
import json
import gzip
import subprocess
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm import tqdm


## 1. Download input files

We use `curl -L` (follows redirects) and skip downloads if the file already exists.


In [8]:
URLS = {
    "tcga_expr": ("https://toil.xenahubs.net/download/tcga_RSEM_gene_tpm.gz", "tcga_RSEM_gene_tpm.gz"),
    "tcga_pheno": ("https://toil.xenahubs.net/download/TcgaTargetGTEX_phenotype.txt.gz", "TcgaTargetGTEX_phenotype.txt.gz"),
    "reactome_rel": ("https://reactome.org/download/current/ReactomePathwaysRelation.txt", "ReactomePathwaysRelation.txt"),
    "reactome_map": ("https://reactome.org/download/current/Ensembl2Reactome_All_Levels.txt", "Ensembl2Reactome_All_Levels.txt"),
}

def download(url: str, out_path: Path) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"[skip] {out_path.name} already exists ({out_path.stat().st_size/1e6:.1f} MB)")
        return out_path

    print(f"[download] {out_path.name}")
    cmd = ["curl", "-L", "-o", str(out_path), url]
    subprocess.run(cmd, check=True)
    print(f"[done] {out_path.name} ({out_path.stat().st_size/1e6:.1f} MB)")
    return out_path

paths = {}
for key, (url, fname) in URLS.items():
    paths[key] = download(url, DATA_DIR / fname)

paths

[download] tcga_RSEM_gene_tpm.gz
[done] tcga_RSEM_gene_tpm.gz (740.8 MB)
[download] TcgaTargetGTEX_phenotype.txt.gz
[done] TcgaTargetGTEX_phenotype.txt.gz (0.1 MB)
[download] ReactomePathwaysRelation.txt
[done] ReactomePathwaysRelation.txt (0.6 MB)
[download] Ensembl2Reactome_All_Levels.txt
[done] Ensembl2Reactome_All_Levels.txt (490.5 MB)


{'tcga_expr': PosixPath('/content/drive/MyDrive/binn_gnn_tcga/data/tcga_RSEM_gene_tpm.gz'),
 'tcga_pheno': PosixPath('/content/drive/MyDrive/binn_gnn_tcga/data/TcgaTargetGTEX_phenotype.txt.gz'),
 'reactome_rel': PosixPath('/content/drive/MyDrive/binn_gnn_tcga/data/ReactomePathwaysRelation.txt'),
 'reactome_map': PosixPath('/content/drive/MyDrive/binn_gnn_tcga/data/Ensembl2Reactome_All_Levels.txt')}

## 2. Sanity checks: file headers and formats

We check:
- expression matrix orientation (genes x samples)
- phenotype header exists
- Reactome files look like tabular text


In [9]:
def head_gz(path: Path, n: int = 2, max_chars: int = 200):
    print(f"\n=== {path.name} (first {n} lines) ===")
    with gzip.open(path, "rt") as f:
        for _ in range(n):
            line = next(f)
            print(line[:max_chars].rstrip(), " ...")

head_gz(paths["tcga_expr"], n=2)
head_gz(paths["tcga_pheno"], n=2)

print("\nReactomePathwaysRelation.txt head:")
print(pd.read_csv(paths["reactome_rel"], sep="\t", header=None, nrows=3))

print("\nEnsembl2Reactome_All_Levels.txt head:")
print(pd.read_csv(paths["reactome_map"], sep="\t", header=None, nrows=3))



=== tcga_RSEM_gene_tpm.gz (first 2 lines) ===
sample	TCGA-19-1787-01	TCGA-S9-A7J2-01	TCGA-G3-A3CH-11	TCGA-EK-A2RE-01	TCGA-44-6778-01	TCGA-F4-6854-01	TCGA-AB-2863-03	TCGA-C8-A1HL-01	TCGA-EW-A2FS-01	TCGA-IR-A3L7-01	TCGA-05-4420-01	TCGA-R6-A8WC-01	T  ...
ENSG00000242268.2	-9.9658	0.2998	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-0.6873	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-9.9658	-0.8599	-0.4521  ...

=== TcgaTargetGTEX_phenotype.txt.gz (first 2 lines) ===
sample	detailed_category	primary disease or tissue	_primary_site	_sample_type	_gender	_study  ...
TCGA-V4-A9EE-01	Uveal Melanoma	Uveal Melanoma	Eye	Primary Tumor	Male	TCGA  ...

ReactomePathwaysRelation.txt head:
              0              1
0  R-BTA-109581   R-BTA-109606
1  R-BTA-109581   R-BTA-169911
2  R-BTA-109581  R-BTA-5357769

Ensembl2Reactome_All_Levels.txt head:
            0              1  \
0  2RSSE.1b.1   R-CEL-162582   
1  2RSSE.1b.1   R-CEL-194315   
2  2RSSE.1b.

## 3. Reactome mapping: ENSG genes → R-HSA pathways

The Reactome mapping file contains multiple species. We keep:
- `species == Homo sapiens`
- `pathway_id` starts with `R-HSA-`
- `ensembl_or_id` starts with `ENSG` (needed for TCGA expression)


In [10]:
reactome_map = pd.read_csv(paths["reactome_map"], sep="\t", header=None)
reactome_map.columns = ["ensembl_or_id", "pathway_id", "url", "pathway_name", "evidence", "species"]

reactome_map["species_clean"] = reactome_map["species"].astype(str).str.strip().str.lower()
reactome_map["pathway_id"] = reactome_map["pathway_id"].astype(str)
reactome_map["ensembl_or_id"] = reactome_map["ensembl_or_id"].astype(str)

g2p = reactome_map[
    reactome_map["species_clean"].eq("homo sapiens")
    & reactome_map["pathway_id"].str.startswith("R-HSA-")
    & reactome_map["ensembl_or_id"].str.startswith("ENSG")
].copy()

g2p = g2p[["ensembl_or_id", "pathway_id"]].drop_duplicates()

print("Gene→Pathway edges:", g2p.shape[0])
print("Unique ENSG genes:", g2p["ensembl_or_id"].nunique())
print("Unique pathways:", g2p["pathway_id"].nunique())
print("Example rows:")
g2p.head()

Gene→Pathway edges: 153961
Unique ENSG genes: 13029
Unique pathways: 2799
Example rows:


,ensembl_or_id,pathway_id
851050,ENSG00000000419,R-HSA-162699
851051,ENSG00000000419,R-HSA-163125
851052,ENSG00000000419,R-HSA-1643685
851053,ENSG00000000419,R-HSA-3781865
851054,ENSG00000000419,R-HSA-392499


## 4. Reactome pathway hierarchy: parent → child (in the file)

Reactome’s `ReactomePathwaysRelation.txt` is a pathway hierarchy edge list.

- **Column 1**: parent pathway (more general)
- **Column 2**: child pathway (more specific)

For the models we want later, it is usually more convenient to use **directed edges child → parent** (bottom-up information flow).  
So we will load the file in its native order (parent, child) but build our graph edges in the **child → parent** direction.

We also keep only human pathways (`R-HSA-`).


In [11]:
rel = pd.read_csv(paths["reactome_rel"], sep="\t", header=None, names=["parent_pathway", "child_pathway"])
rel["child_pathway"] = rel["child_pathway"].astype(str)
rel["parent_pathway"] = rel["parent_pathway"].astype(str)

rel = rel[
    rel["child_pathway"].str.startswith("R-HSA-")
    & rel["parent_pathway"].str.startswith("R-HSA-")
].drop_duplicates()

print("Human pathway relations:", rel.shape[0])
rel.head()

Human pathway relations: 2864


,child_pathway,parent_pathway
11142,R-HSA-109581,R-HSA-109606
11143,R-HSA-109581,R-HSA-169911
11144,R-HSA-109581,R-HSA-5357769
11145,R-HSA-109581,R-HSA-75153
11146,R-HSA-109582,R-HSA-140877


## 5. Load phenotype and create Tumor vs Normal labels

The phenotype file may contain non-UTF8 bytes, so we use a fallback encoding strategy.


In [12]:
def read_tsv_gz(path: Path, sep: str = "\t", encodings=("utf-8", "latin1", "cp1252")) -> pd.DataFrame:
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, sep=sep, compression="gzip", encoding=enc, low_memory=False)
        except UnicodeDecodeError as e:
            last_err = e
            print(f"[warn] failed with encoding={enc}: {e}")
    raise last_err

pheno = read_tsv_gz(paths["tcga_pheno"], sep="\t")
print("Phenotype shape:", pheno.shape)
print("Columns:", pheno.columns.tolist())

pheno_tcga = pheno[pheno["_study"] == "TCGA"].copy()

keep_types = {"Primary Tumor": 1, "Solid Tissue Normal": 0}
pheno_tcga = pheno_tcga[pheno_tcga["_sample_type"].isin(keep_types)].copy()
pheno_tcga["y"] = pheno_tcga["_sample_type"].map(keep_types).astype(int)

print("TCGA tumor+normal samples:", pheno_tcga.shape[0])
print("Tumor:", (pheno_tcga["y"]==1).sum(), "Normal:", (pheno_tcga["y"]==0).sum())

pheno_tcga[["sample", "_sample_type", "y"]].head()

[warn] failed with encoding=utf-8: 'utf-8' codec can't decode byte 0xca in position 11: invalid continuation byte
Phenotype shape: (19131, 7)
Columns: ['sample', 'detailed_category', 'primary disease or tissue', '_primary_site', '_sample_type', '_gender', '_study']
TCGA tumor+normal samples: 9912
Tumor: 9185 Normal: 727


,sample,_sample_type,y
0,TCGA-V4-A9EE-01,Primary Tumor,1
1,TCGA-VD-AA8N-01,Primary Tumor,1
2,TCGA-V4-A9EI-01,Primary Tumor,1
3,TCGA-VD-AA8O-01,Primary Tumor,1
4,TCGA-WC-A888-01,Primary Tumor,1


## 6. Match phenotype samples to expression columns

The expression file is genes x samples. The header line gives us the sample columns.


In [13]:
with gzip.open(paths["tcga_expr"], "rt") as f:
    header = f.readline().strip().split("\t")

gene_col_name = header[0]           # usually "sample" (holds gene ids)
expr_sample_cols = header[1:]
expr_samples = set(expr_sample_cols)

print("Expression columns:", len(expr_samples))
print("Gene id column name:", gene_col_name)

pheno_tcga = pheno_tcga[pheno_tcga["sample"].isin(expr_samples)].copy()

print("After matching expression columns:", pheno_tcga.shape[0])
print("Tumor:", (pheno_tcga["y"]==1).sum(), "Normal:", (pheno_tcga["y"]==0).sum())

Expression columns: 10535
Gene id column name: sample
After matching expression columns: 9912
Tumor: 9185 Normal: 727


## 7. Load expression filtered to Reactome genes (streaming)

We stream-load `tcga_RSEM_gene_tpm.gz` in chunks, keeping:
- labeled samples (from phenotype)
- genes that exist in `g2p` (Reactome-mapped ENSG ids)

Identifier harmonization:
- TCGA uses `ENSG... .<version>` (example: `ENSG00000141510.12`)
- Reactome uses versionless ENSG ids
So we strip the suffix `.12` using a simple regex.


In [14]:
def strip_ensembl_version(x: str) -> str:
    return re.sub(r"\.\d+$", "", str(x))

reactome_gene_set = set(g2p["ensembl_or_id"].astype(str))
keep_samples = pheno_tcga["sample"].tolist()

# Optional: for faster runs while debugging, limit number of samples
MAX_SAMPLES = None  # e.g. 2000
if MAX_SAMPLES is not None and len(keep_samples) > MAX_SAMPLES:
    keep_samples = keep_samples[:MAX_SAMPLES]
    pheno_tcga = pheno_tcga[pheno_tcga["sample"].isin(keep_samples)].copy()

usecols = [gene_col_name] + keep_samples
chunksize = 3000

expr_chunks = []
for chunk in pd.read_csv(paths["tcga_expr"], sep="\t", usecols=usecols, chunksize=chunksize):
    chunk = chunk.rename(columns={gene_col_name: "gene_id_raw"})
    chunk["gene_id"] = chunk["gene_id_raw"].map(strip_ensembl_version)

    chunk = chunk[chunk["gene_id"].isin(reactome_gene_set)]
    if chunk.empty:
        continue

    chunk = chunk.drop(columns=["gene_id_raw"]).set_index("gene_id")
    expr_chunks.append(chunk)

expr = pd.concat(expr_chunks, axis=0)

# Align columns and labels
expr = expr.loc[:, pheno_tcga["sample"]]
y = pheno_tcga.set_index("sample").loc[expr.columns, "y"].astype(int)

print("Expression shape (genes x samples):", expr.shape)
print("Label counts:", y.value_counts().to_dict())
print("Tumor fraction:", float(y.mean()))

expr.iloc[:3, :5]

Expression shape (genes x samples): (11403, 9912)
Label counts: {1: 9185, 0: 727}
Tumor fraction: 0.9266545601291364


,TCGA-V4-A9EE-01,TCGA-VD-AA8N-01,TCGA-V4-A9EI-01,TCGA-VD-AA8O-01,TCGA-WC-A888-01
gene_id,,,,,
ENSG00000167578,5.4611,4.8334,5.2597,5.9081,6.3705
ENSG00000078237,3.5473,2.8178,3.5633,2.7826,2.3732
ENSG00000198242,10.5737,11.0476,10.4901,10.8085,10.5106


## 8. Save dataset artifacts

These files are the starting point for training later.


In [15]:
expr_path = OUT_DIR / "expr_reactome_tcga_tumor_normal.parquet"
y_path = OUT_DIR / "y_tcga_tumor_normal.csv"

expr.to_parquet(expr_path)
y.to_csv(y_path)

print("Saved:")
print(" -", expr_path)
print(" -", y_path)

Saved:
 - /content/drive/MyDrive/binn_gnn_tcga/outputs/expr_reactome_tcga_tumor_normal.parquet
 - /content/drive/MyDrive/binn_gnn_tcga/outputs/y_tcga_tumor_normal.csv


## 9. Build the shared graph (genes + pathways)

Nodes:
- Gene nodes: `expr.index` (filtered ENSG ids)
- Pathway nodes: pathways connected to the kept genes, plus all ancestors in the hierarchy

Edges:
- gene → pathway (from `g2p`)
- pathway(child) → pathway(parent) (from `rel`)

We also sort edges to make the saved graph deterministic across runs.


In [16]:
# Gene nodes come from the prepared expression matrix
gene_nodes = sorted(expr.index.astype(str).tolist())
genes_in_expr = set(gene_nodes)

# Filter gene→pathway edges to kept genes
g2p_f = g2p[g2p["ensembl_or_id"].astype(str).isin(genes_in_expr)].copy()
g2p_f = g2p_f.drop_duplicates()

# Closure: include all ancestor pathways
parents_map = rel.groupby("child_pathway")["parent_pathway"].apply(list).to_dict()
direct_pathways = set(g2p_f["pathway_id"].astype(str))

pathways_all = set(direct_pathways)
stack = list(direct_pathways)
while stack:
    child = stack.pop()
    for parent in parents_map.get(child, []):
        if parent not in pathways_all:
            pathways_all.add(parent)
            stack.append(parent)

pathway_nodes = sorted(pathways_all)

print("Genes:", len(gene_nodes))
print("Pathways (closure):", len(pathway_nodes))

# Node ids: genes first, then pathways
all_nodes = gene_nodes + pathway_nodes
node2id = {n: i for i, n in enumerate(all_nodes)}

num_genes = len(gene_nodes)
num_nodes = len(all_nodes)

# Build sorted edge tables
g2p_edges = g2p_f[g2p_f["pathway_id"].isin(pathways_all)][["ensembl_or_id", "pathway_id"]].drop_duplicates()
g2p_edges = g2p_edges.sort_values(["ensembl_or_id", "pathway_id"])

rel_f = rel[rel["child_pathway"].isin(pathways_all) & rel["parent_pathway"].isin(pathways_all)].drop_duplicates()
rel_f = rel_f.sort_values(["child_pathway", "parent_pathway"])

# Convert to integer ids
src_gene = g2p_edges["ensembl_or_id"].map(node2id).to_numpy(dtype=np.int64)
dst_gene = g2p_edges["pathway_id"].map(node2id).to_numpy(dtype=np.int64)

src_pw = rel_f["child_pathway"].map(node2id).to_numpy(dtype=np.int64)
dst_pw = rel_f["parent_pathway"].map(node2id).to_numpy(dtype=np.int64)

src = np.concatenate([src_gene, src_pw])
dst = np.concatenate([dst_gene, dst_pw])

edge_index = np.vstack([src, dst])
num_edges = edge_index.shape[1]

is_gene = np.zeros(num_nodes, dtype=bool)
is_gene[:num_genes] = True

print("Total nodes:", num_nodes)
print("Total edges:", num_edges)
print("Gene nodes:", int(is_gene.sum()), "Pathway nodes:", int((~is_gene).sum()))

edge_index[:, :5]

Genes: 11403
Pathways (closure): 2848
Total nodes: 14251
Total edges: 140631
Gene nodes: 11403 Pathway nodes: 2848


array([[    0,     0,     0,     0,     0],
       [11652, 11658, 11676, 12317, 12395]])

## 10. Save graph artifacts

We save:
- tensors for training (`edge_index.pt`, `is_gene.pt`)
- lookup tables (`node_table.csv`, `edge_table.csv`, `node2id.json`)
- metadata for reproducibility (`metadata.json`)


In [17]:
import torch

node_table = pd.DataFrame({
    "node_id": np.arange(num_nodes, dtype=int),
    "node_name": all_nodes,
    "node_type": ["gene"] * num_genes + ["pathway"] * (num_nodes - num_genes),
})

edge_types = (["gene_to_pathway"] * len(src_gene)) + (["pathway_to_parent"] * len(src_pw))
edge_table = pd.DataFrame({
    "src": src,
    "dst": dst,
    "src_name": [all_nodes[i] for i in src],
    "dst_name": [all_nodes[i] for i in dst],
    "edge_type": edge_types,
})

edge_index_t = torch.tensor(edge_index, dtype=torch.long)
is_gene_t = torch.tensor(is_gene, dtype=torch.bool)

torch.save(edge_index_t, GRAPH_DIR / "edge_index.pt")
torch.save(is_gene_t, GRAPH_DIR / "is_gene.pt")

node_table.to_csv(GRAPH_DIR / "node_table.csv", index=False)
edge_table.to_csv(GRAPH_DIR / "edge_table.csv", index=False)

with open(GRAPH_DIR / "node2id.json", "w") as f:
    json.dump(node2id, f)

metadata = {
    "created_at": datetime.now().isoformat(),
    "num_nodes": int(num_nodes),
    "num_edges": int(num_edges),
    "num_genes": int(num_genes),
    "num_pathways": int(num_nodes - num_genes),
    "expr_shape": [int(expr.shape[0]), int(expr.shape[1])],
    "label_counts": y.value_counts().to_dict(),
    "files": {k: str(v) for k, v in paths.items()},
    "params": {
        "task": "tumor_vs_normal",
        "max_samples": MAX_SAMPLES,
        "chunksize": chunksize,
    },
}

with open(GRAPH_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved graph artifacts to:", GRAPH_DIR)
for p in sorted(GRAPH_DIR.glob("*")):
    print(" -", p.name)

Saved graph artifacts to: /content/drive/MyDrive/binn_gnn_tcga/outputs/graph
 - edge_index.pt
 - edge_table.csv
 - is_gene.pt
 - metadata.json
 - node2id.json
 - node_table.csv


## 11. Done

You can now start a training notebook by loading:

- `outputs/expr_reactome_tcga_tumor_normal.parquet`
- `outputs/y_tcga_tumor_normal.csv`
- `outputs/graph/edge_index.pt`
- `outputs/graph/is_gene.pt`
- `outputs/graph/node_table.csv`

No re-downloading or re-processing is needed unless you want to change the task or filtering.


# Phase 2: Build a BINN-style layered graph (no training yet)

In Phase 1 we created a **raw biological graph**:

- **Genes** (observed features)
- **Pathways** (hidden nodes)
- Edges:
  - gene → pathway (membership from Reactome mapping)
  - pathway → parent pathway (Reactome hierarchy, used bottom-up)

That raw graph is ideal for a “standard GNN” (GCN/GAT) because GNNs can ingest an edge list directly.

However, the original BINN is a **feedforward masked network** where each layer only connects to the next layer.  
So Phase 2 creates a *layered* version of the same biology that we can later use for a **BINN-exact message passing schedule**.

## What Phase 2 produces

We will create a new folder:

- `outputs/graph_layered_binn/`

It will contain:

- `node_table_layered.csv` (layered nodes, including padded duplicates)
- `edge_table_layered.csv` (edges only between consecutive layers)
- `edge_index_layered.pt` (PyTorch tensor for later PyG use)
- `layer_info.json` (layer sizes, leaf and root nodes, etc.)

## Key idea: “raggedness” and padding

Reactome’s pathway hierarchy is a DAG with uneven depth. That creates **edges that jump multiple levels**.

To make a strict feedforward layer-to-layer network, we will **pad** these jumps by creating *duplicate copies of pathways at intermediate layers* so that every edge goes from layer ℓ → ℓ+1.


## A. Load the raw graph artifacts (from Phase 1)

We reuse what we already saved in `outputs/graph/`:

- `node_table.csv`
- `edge_table.csv`

We also run a quick sanity check to confirm the pathway hierarchy direction is correct (child → parent).


In [25]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import torch

RAW_GRAPH_DIR = GRAPH_DIR
LAYERED_GRAPH_DIR = OUT_DIR / "graph_layered_binn"
LAYERED_GRAPH_DIR.mkdir(parents=True, exist_ok=True)

node_table = pd.read_csv(RAW_GRAPH_DIR / "node_table.csv")
edge_table = pd.read_csv(RAW_GRAPH_DIR / "edge_table.csv")

num_nodes = node_table.shape[0]
num_edges = edge_table.shape[0]
num_genes = int((node_table["node_type"] == "gene").sum())
num_pathways = num_nodes - num_genes

print(f"Raw graph loaded: N={num_nodes} (genes={num_genes}, pathways={num_pathways}), E={num_edges}")
print(edge_table["edge_type"].value_counts())

# Pathway hierarchy edges (expected direction: child → parent)
pw_edges = edge_table.loc[edge_table["edge_type"].eq("pathway_to_parent"), ["src", "dst"]].astype(int)
pw_src = pw_edges["src"].to_numpy()
pw_dst = pw_edges["dst"].to_numpy()

root_candidates = set(pw_dst) - set(pw_src)  # parent pathways that never appear as child
print("Candidate root pathways (dst not in src):", len(root_candidates))

if len(root_candidates) > 300:
    print("\nWARNING: root count is very large.")
    print("This usually means the pathway hierarchy direction is reversed.")
    print("Fix: rerun Phase 1 starting at Section 4 (Reactome relations) and rebuild the raw graph artifacts.\n")


Saved corrected graph to: /content/drive/MyDrive/binn_gnn_tcga/outputs/graph_fixed


In [26]:
from collections import deque

# Identify node IDs
gene_ids = node_table.loc[node_table["node_type"].eq("gene"), "node_id"].astype(int).to_numpy()
pathway_ids = node_table.loc[node_table["node_type"].eq("pathway"), "node_id"].astype(int).to_numpy()

# Build indegree = number of children (incoming from child pathways)
# Leaves (bottom pathways) have indegree 0 in the pathway-only graph.
indeg = np.zeros(num_nodes, dtype=np.int32)
np.add.at(indeg, pw_dst, 1)
indeg0 = indeg.copy()  # keep a copy to identify leaves later

# Adjacency: child -> [parents]
parents = {}
for s, d in zip(pw_src, pw_dst):
    parents.setdefault(int(s), []).append(int(d))

# Longest-path level from leaves (Kahn + DP)
level_base = np.zeros(num_nodes, dtype=np.int32)
q = deque([int(p) for p in pathway_ids if indeg[int(p)] == 0])

seen = 0
while q:
    u = q.popleft()
    seen += 1
    for p in parents.get(u, []):
        if level_base[p] < level_base[u] + 1:
            level_base[p] = level_base[u] + 1
        indeg[p] -= 1
        if indeg[p] == 0:
            q.append(p)

if seen != len(pathway_ids):
    print("WARNING: pathway graph may contain cycles or disconnected parts (unexpected for Reactome).")
    print("Computed levels for", seen, "of", len(pathway_ids), "pathway nodes.")

# Pathway layers start at 1 so we can reserve layer 0 for genes
pathway_layer = level_base + 1
L = int(pathway_layer[pathway_ids].max())

leaf_pathways = set(int(p) for p in pathway_ids if indeg0[int(p)] == 0)
root_pathways = sorted(root_candidates)

print("Max pathway layer (L):", L)
print("Leaf pathways:", len(leaf_pathways))
print("Root pathways:", len(root_pathways))

# Show a few root pathway names for sanity
if len(root_pathways) > 0:
    preview = node_table.loc[node_table["node_id"].isin(root_pathways[:10]), ["node_name"]]
    display(preview)


N: 14251 E: 140631 num_genes: 11403


## B. Create layered nodes (duplicate pathways to remove raggedness)

We keep genes only in **layer 0**.

Pathways have an intrinsic “base layer” from the hierarchy depth (leaf pathways start at layer 1).

To ensure a strict feedforward structure, we may need to **duplicate** a pathway across multiple layers:

- If a child pathway connects to a high-level parent, we create copies of the child pathway at intermediate layers.
- Root pathways (no parent) are padded up to the maximum layer `L` so we can read out predictions from a single layer.

The result is a new node set where each node is `(original_node_id, layer)`.


In [27]:
# Compute how far upward each pathway must be duplicated.
# For an edge (child -> parent), the child must exist at layer (layer(parent) - 1).
max_required_layer = pathway_layer.copy()

for s, d in zip(pw_src, pw_dst):
    req = int(pathway_layer[int(d)] - 1)
    if req > max_required_layer[int(s)]:
        max_required_layer[int(s)] = req

# Pad all root pathways up to the global maximum layer L so we can read out from one layer.
for p in root_pathways:
    max_required_layer[int(p)] = L

# Convenience lookup: original node_id -> name
name_by_id = dict(zip(node_table["node_id"].astype(int), node_table["node_name"].astype(str)))

# Create layered nodes in layer order:
# - Layer 0: all genes
# - Layer 1..L: pathways, including duplicates where needed
layered_lookup = {}  # (orig_id, layer) -> layered_id
layered_records = []

next_id = 0

# Genes (layer 0)
for gid in gene_ids:
    gid = int(gid)
    layered_lookup[(gid, 0)] = next_id
    layered_records.append(
        dict(
            layered_id=next_id,
            orig_id=gid,
            orig_name=name_by_id[gid],
            node_type="gene",
            layer=0,
            is_duplicate=False,
        )
    )
    next_id += 1

# Pathways (layers 1..L)
pathway_ids_sorted = np.sort(pathway_ids.astype(int))
for layer in range(1, L + 1):
    for pid in pathway_ids_sorted:
        if pathway_layer[pid] <= layer <= max_required_layer[pid]:
            layered_lookup[(pid, layer)] = next_id
            layered_records.append(
                dict(
                    layered_id=next_id,
                    orig_id=pid,
                    orig_name=name_by_id[pid],
                    node_type="pathway",
                    layer=layer,
                    is_duplicate=bool(layer > pathway_layer[pid]),
                )
            )
            next_id += 1

node_table_layered = pd.DataFrame(layered_records)
print("Layered node_table:", node_table_layered.shape)

layer_counts = node_table_layered["layer"].value_counts().sort_index()
print("Nodes per layer:")
print(layer_counts)


Root pathways: 2014


## C. Create layered edges (all edges go from layer ℓ → ℓ+1)

We create three types of edges:

1. **Gene → leaf pathway** edges (layer 0 → 1)
2. **Pathway child → pathway parent** edges (layer k → k+1), aligned to the parent’s layer
3. **Padding edges** connecting duplicates of the same pathway across layers (layer k → k+1)

After this step we verify that every edge increases the layer by exactly 1.


In [28]:
# Build layered edges (all must be layer ℓ -> ℓ+1)
src_list = []
dst_list = []
etype_list = []

# 1) Gene -> leaf pathway edges (0 -> 1)
g2p = edge_table.loc[edge_table["edge_type"].eq("gene_to_pathway"), ["src", "dst"]].astype(int)

# Keep only edges to leaf pathways so genes connect to the bottom layer
g2p = g2p[g2p["dst"].isin(leaf_pathways)].copy()
print("Gene -> leaf pathway edges:", g2p.shape[0])

for s, d in g2p.itertuples(index=False):
    src_list.append(layered_lookup[(int(s), 0)])
    dst_list.append(layered_lookup[(int(d), 1)])
    etype_list.append("gene_to_leaf_pathway")

# 2) Pathway child -> pathway parent edges, aligned to the parent's layer
for s, d in zip(pw_src, pw_dst):
    s = int(s)
    d = int(d)
    parent_layer = int(pathway_layer[d])
    src_layer = parent_layer - 1
    dst_layer = parent_layer

    src_list.append(layered_lookup[(s, src_layer)])
    dst_list.append(layered_lookup[(d, dst_layer)])
    etype_list.append("pathway_child_to_parent")

# 3) Padding edges for duplicates of the same pathway across layers
#    (pid, k) -> (pid, k+1)
for pid in pathway_ids_sorted:
    start = int(pathway_layer[pid])
    end = int(max_required_layer[pid])
    for layer in range(start, end):
        src_list.append(layered_lookup[(pid, layer)])
        dst_list.append(layered_lookup[(pid, layer + 1)])
        etype_list.append("padding_identity")

src_arr = np.array(src_list, dtype=np.int64)
dst_arr = np.array(dst_list, dtype=np.int64)

edge_table_layered = pd.DataFrame({"src": src_arr, "dst": dst_arr, "edge_type": etype_list})
edge_index_layered = torch.tensor(np.vstack([src_arr, dst_arr]), dtype=torch.long)

# Validate layer deltas
layer_by_layered_id = node_table_layered.sort_values("layered_id")["layer"].to_numpy()
deltas = layer_by_layered_id[dst_arr] - layer_by_layered_id[src_arr]
unique_deltas, delta_counts = np.unique(deltas, return_counts=True)
print("Edge delta counts (dst_layer - src_layer):", dict(zip(unique_deltas.tolist(), delta_counts.tolist())))

assert np.all(deltas == 1), "Found edges that do not go from layer ℓ to layer ℓ+1"

# Readout nodes: root pathways at the final layer L
root_layered_ids = [layered_lookup[(int(p), L)] for p in root_pathways]
print("Root pathways at final layer:", len(root_layered_ids))

# Quick per-layer edge counts (by destination layer)
dst_layers = layer_by_layered_id[dst_arr]
edge_counts_by_layer = pd.Series(dst_layers).value_counts().sort_index()
print("Edges per layer (destination layer):")
print(edge_counts_by_layer)


Max level: 12


## D. Save the layered graph artifacts

These are the only files you will need later when you start model training.

We save:

- `node_table_layered.csv`
- `edge_table_layered.csv`
- `edge_index_layered.pt`
- `layer_info.json`


In [29]:
# Save layered artifacts (for training later)
node_table_layered.to_csv(LAYERED_GRAPH_DIR / "node_table_layered.csv", index=False)
edge_table_layered.to_csv(LAYERED_GRAPH_DIR / "edge_table_layered.csv", index=False)
torch.save(edge_index_layered, LAYERED_GRAPH_DIR / "edge_index_layered.pt")

layer_info = {
    "created_at": datetime.now().isoformat(),
    "L": int(L),
    "num_layered_nodes": int(node_table_layered.shape[0]),
    "num_layered_edges": int(edge_table_layered.shape[0]),
    "nodes_per_layer": {int(k): int(v) for k, v in layer_counts.to_dict().items()},
    "edges_per_dst_layer": {int(k): int(v) for k, v in edge_counts_by_layer.to_dict().items()},
    "num_leaf_pathways": int(len(leaf_pathways)),
    "num_root_pathways": int(len(root_pathways)),
    "root_pathways_orig_ids": [int(p) for p in root_pathways],
    "root_pathways_layered_ids": [int(i) for i in root_layered_ids],
}

with open(LAYERED_GRAPH_DIR / "layer_info.json", "w") as f:
    json.dump(layer_info, f, indent=2)

print("\nSaved layered graph to:", LAYERED_GRAPH_DIR)
for p in sorted(LAYERED_GRAPH_DIR.glob("*")):
    print(" -", p.name)


Edge delta counts (dst_level - src_level):
{1: 25565, 2: 27800, 3: 31143, 4: 28765, 5: 17602, 6: 5869, 7: 2417, 8: 974, 9: 353, 10: 91, 11: 34, 12: 18}
Using layers 1..L = 12
Layer 1: edges 22726
Layer 2: edges 187
Layer 3: edges 498
Layer 4: edges 906
Layer 5: edges 792
Layer 6: edges 281
Layer 7: edges 106
Layer 8: edges 48
Layer 9: edges 14
Layer 10: edges 3
Layer 11: edges 3
Layer 12: edges 1


## E. Sanity checks and next step

Sanity checks you should expect:

- **Root pathways** should be a relatively small number (tens, not thousands).
- **Edge delta counts** should be `{1: ...}` only.
- **Edges per layer** should be non-zero for most layers (because padding edges fill gaps).

If any of these checks fail, fix the raw graph direction first (Phase 1, Section 4).

### Next step (separate notebook)

Now we can implement the **BINN-exact MPNN layer**:

- learn a **unique scalar weight per edge**
- perform message passing from layer 0 to layer L
- read out from `root_pathways_layered_ids` (not global pooling)

We will do that next, without starting training yet.


In [23]:
# Optional: inspect a few duplicated pathways (to understand padding)

dups = node_table_layered[node_table_layered["is_duplicate"]].copy()
print("Number of duplicate pathway-nodes:", dups.shape[0])

if dups.shape[0] > 0:
    sample_dups = dups.sample(min(10, dups.shape[0]), random_state=0)[["orig_name", "layer"]]
    display(sample_dups.sort_values(["orig_name", "layer"]))


Saved layer artifacts to: /content/drive/MyDrive/binn_gnn_tcga/outputs/binn_layered
